# Final normal-estimation search analysis

This notebook summarizes the completed explainable normal-estimation search from canonical records. It is designed for hundreds of iterations: discarded attempts are rendered as a low-alpha density cloud, while only retained milestones and the final frontier are emphasized.

The final benchmark section reads `experiments/normal-estimation/final-results.json`, produced once from the frozen finalist on the complete official PCPNet test list. **Development and validation scores are for research diagnostics; only the final test row is directly comparable with published PCPNet benchmark rows.** The official test list includes the validation geometries, so it is not a strictly disjoint third split.


In [ ]:
from __future__ import annotations

import json
import textwrap
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

%config InlineBackend.figure_format = "retina"
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "experiments"
        ).is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the point-cloud-central repository root")


def metric(record: dict[str, object], key: str) -> float:
    value = record.get(key)
    return np.nan if value is None else float(value)


def markdown_table(headers: list[str], rows: list[list[str]]) -> str:
    header = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join(["---"] + ["---:"] * (len(headers) - 1)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return "\n".join([header, separator, *body])


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXPERIMENT_DIR = REPO_ROOT / "experiments" / "normal-estimation"
RECORD_DIR = EXPERIMENT_DIR / "records"
FINAL_RESULTS_PATH = EXPERIMENT_DIR / "final-results.json"
CONDITIONS = ("clean", "low", "medium", "high", "stripe", "gradient")
CONDITION_LABELS = {
    "clean": "None",
    "low": "Low",
    "medium": "Medium",
    "high": "High",
    "stripe": "Stripe",
    "gradient": "Gradient",
}
STATUS_COLORS = {
    "keep": "#1b9e77",
    "provisional": "#e6ab02",
    "discard": "#9aa0a6",
    "crash": "#d95f02",
}
PUBLISHED = {
    "PCA (published)": (12.29, 12.87, 18.38, 27.52, 13.66, 12.81, 16.25),
    "MSECNet": (3.84, 8.74, 16.10, 21.05, 4.34, 4.51, 9.76),
    "PFF-Net": (3.32, 8.34, 15.63, 20.94, 4.10, 3.92, 9.38),
}

record_paths = sorted(RECORD_DIR.glob("[0-9][0-9][0-9][0-9].json"))
records = [json.loads(path.read_text()) for path in record_paths]
if not records:
    raise FileNotFoundError(f"No canonical records found in {RECORD_DIR}")
if [record["iteration"] for record in records] != list(range(len(records))):
    raise RuntimeError("Canonical records are not contiguous")
if not FINAL_RESULTS_PATH.is_file():
    raise FileNotFoundError(
        f"Missing frozen test result: {FINAL_RESULTS_PATH}. Do not substitute validation metrics."
    )
final_results = json.loads(FINAL_RESULTS_PATH.read_text())

latest = records[-1]
validated_commit = latest["validated"]["commit"]
final_record = next(
    record for record in records if record.get("candidate_commit") == validated_commit
)
if final_results["candidate_commit"] != validated_commit:
    raise RuntimeError("Frozen test results do not match the validated frontier")
if final_results["estimator_sha256"] != final_record["estimator_sha256"]:
    raise RuntimeError("Frozen test results do not match the finalist estimator hash")

iterations = np.array([int(record["iteration"]) for record in records])
development = np.array([metric(record, "dev_rmse") for record in records])
validation = np.array([metric(record, "val_rmse") for record in records])
statuses = np.array([str(record["status"]) for record in records])
measured_mask = np.isfinite(development)
keep_records = [record for record in records if record["status"] == "keep"]
validated_records = [record for record in records if record.get("val_rmse") is not None]
status_counts = Counter(statuses)

print(
    f"Loaded {len(records)} completed experiments (#{records[0]['iteration']}-#{latest['iteration']})."
)
print(
    f"Measured: {measured_mask.sum()} | Validated: {len(validated_records)} | "
    f"Kept: {len(keep_records)} | Crashes: {status_counts['crash']}"
)
print(
    f"Frozen finalist: #{final_record['iteration']} {validated_commit[:7]} | "
    f"test RMSE={final_results['rmse']:.6f}°"
)

## Search summary and retained milestones

The compact table reports only the 12 retained frontier changes, rather than labeling all 323 attempts. Runtime is measured on the development tier and excludes prepared neighbor search.


In [ ]:
summary_rows = [
    [
        "Completed iterations",
        str(len(records)),
    ],
    ["Measured candidates", str(int(measured_mask.sum()))],
    ["Validation evaluations", str(len(validated_records))],
    ["Retained milestones", str(len(keep_records))],
    ["Discarded", str(status_counts["discard"])],
    ["Provisional", str(status_counts["provisional"])],
    ["Crashes", str(status_counts["crash"])],
    ["Finalist iteration", f"#{final_record['iteration']}"],
    ["Final development RMSE", f"{final_record['dev_rmse']:.6f}°"],
    ["Final validation RMSE", f"{final_record['val_rmse']:.6f}°"],
    ["Final test RMSE", f"{final_results['rmse']:.6f}°"],
]
display(Markdown(markdown_table(["Summary", "Value"], summary_rows)))

milestone_rows = [
    [
        f"#{record['iteration']}",
        f"{record['dev_rmse']:.3f}",
        f"{record['val_rmse']:.3f}",
        f"{record['runtime_s']:.3f}",
        textwrap.shorten(str(record["description"]), width=72, placeholder="…"),
    ]
    for record in keep_records
]
display(
    Markdown(
        "### Retained frontier milestones\n\n"
        + markdown_table(
            ["Iteration", "Dev RMSE", "Val RMSE", "Runtime (s)", "Change"],
            milestone_rows,
        )
    )
)

## Aggregate development progress

The left panel shows every measured candidate with a robust y-range; extreme failures are clipped rather than stretching the plot. The right panel zooms into the competitive band after the initial exploration. Retained candidates are numbered, avoiding hundreds of overlapping description labels.


In [ ]:
running_best = np.minimum.accumulate(np.where(measured_mask, development, np.inf))
running_best[~np.isfinite(running_best)] = np.nan
keep_iterations = np.array([int(record["iteration"]) for record in keep_records])
keep_dev = np.array([float(record["dev_rmse"]) for record in keep_records])

lower = float(np.quantile(development[measured_mask], 0.005)) - 0.08
upper = float(np.quantile(development[measured_mask], 0.99)) + 0.10
zoom_lower = (
    min(float(np.min(development[measured_mask])), float(final_record["dev_rmse"]))
    - 0.06
)
zoom_upper = min(18.95, upper)

fig, (ax_all, ax_zoom) = plt.subplots(
    1, 2, figsize=(18, 7), sharex=True, layout="constrained"
)
for ax in (ax_all, ax_zoom):
    ax.scatter(
        iterations[measured_mask],
        np.clip(development[measured_mask], lower, upper),
        s=16,
        color=STATUS_COLORS["discard"],
        alpha=0.30,
        linewidths=0,
        label="Measured attempt",
        zorder=1,
    )
    ax.plot(
        iterations,
        running_best,
        color="#4c78a8",
        linewidth=1.8,
        alpha=0.9,
        label="Best measured so far",
        zorder=2,
    )
    ax.step(
        keep_iterations,
        keep_dev,
        where="post",
        color=STATUS_COLORS["keep"],
        linewidth=2.8,
        label="Retained frontier",
        zorder=3,
    )
    ax.scatter(
        keep_iterations,
        keep_dev,
        s=58,
        color=STATUS_COLORS["keep"],
        edgecolors="white",
        linewidths=0.8,
        zorder=4,
    )
    ax.scatter(
        [final_record["iteration"]],
        [final_record["dev_rmse"]],
        marker="*",
        s=230,
        color="#d62728",
        edgecolors="black",
        linewidths=0.6,
        label="Frozen finalist",
        zorder=5,
    )
    ax.set_xlabel("Experiment #")
    ax.set_ylabel("Development RMSE (degrees, lower is better)")
    ax.grid(True, alpha=0.20)

ax_all.set_ylim(lower, upper)
ax_all.set_title("All 305 measured candidates (robust y-range)")
ax_zoom.set_ylim(zoom_lower, zoom_upper)
ax_zoom.set_xlim(0, int(iterations.max()) + 3)
ax_zoom.set_title("Competitive band")
for record in keep_records:
    ax_zoom.annotate(
        f"#{record['iteration']}",
        (int(record["iteration"]), float(record["dev_rmse"])),
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
        fontsize=8,
        color="#0b6e4f",
    )
ax_zoom.legend(loc="upper right", fontsize=9)
fig.suptitle(
    f"Normal-estimation search: {len(records)} iterations, "
    f"{len(keep_records)} retained milestones",
    fontsize=15,
)
plt.show()

## Validation evidence

Only 25 candidates were promoted to validation. The green step line uses retained validation results; gray points are candidates rejected after validation. This separation avoids implying that unvalidated development improvements generalized.


In [ ]:
val_iterations = np.array([int(record["iteration"]) for record in validated_records])
val_scores = np.array([float(record["val_rmse"]) for record in validated_records])
val_statuses = np.array([str(record["status"]) for record in validated_records])
val_keep_mask = val_statuses == "keep"

fig, ax = plt.subplots(figsize=(15, 6), layout="constrained")
ax.scatter(
    val_iterations[~val_keep_mask],
    val_scores[~val_keep_mask],
    s=38,
    color=STATUS_COLORS["discard"],
    alpha=0.65,
    label="Rejected after validation",
)
ax.scatter(
    val_iterations[val_keep_mask],
    val_scores[val_keep_mask],
    s=70,
    color=STATUS_COLORS["keep"],
    edgecolors="white",
    linewidths=0.8,
    label="Retained",
    zorder=3,
)
ax.step(
    val_iterations[val_keep_mask],
    val_scores[val_keep_mask],
    where="post",
    color=STATUS_COLORS["keep"],
    linewidth=2.6,
    label="Retained validation frontier",
)
for record in keep_records:
    ax.annotate(
        f"#{record['iteration']}",
        (int(record["iteration"]), float(record["val_rmse"])),
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
        fontsize=8,
    )
ax.set(
    xlabel="Experiment #",
    ylabel="Validation RMSE (degrees, lower is better)",
    title="Promoted candidates on the validation tier",
)
ax.legend()
ax.grid(True, alpha=0.20)
plt.show()

## Per-condition development trajectories

Each panel shows all measured attempts faintly and only the retained milestones as a connected line. The dashed line is the experiment's fixed-k PCA baseline. Independent y-scales keep improvements visible across conditions with very different error magnitudes.


In [ ]:
condition_records = [
    record
    for record in records
    if isinstance(record.get("condition_rmse"), dict)
    and isinstance(record["condition_rmse"].get("development"), dict)
]
condition_iterations = np.array(
    [int(record["iteration"]) for record in condition_records]
)

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True, layout="constrained")
for ax, condition in zip(axes.flat, CONDITIONS, strict=True):
    scores = np.array(
        [
            float(record["condition_rmse"]["development"][condition])
            for record in condition_records
        ]
    )
    retained_scores = np.array(
        [
            float(record["condition_rmse"]["development"][condition])
            for record in keep_records
        ]
    )
    low, high = np.quantile(scores, [0.01, 0.99])
    margin = max((high - low) * 0.08, 0.08)
    ax.scatter(
        condition_iterations,
        np.clip(scores, low - margin, high + margin),
        s=12,
        color=STATUS_COLORS["discard"],
        alpha=0.25,
        linewidths=0,
    )
    ax.plot(
        keep_iterations,
        retained_scores,
        color=STATUS_COLORS["keep"],
        marker="o",
        markersize=4.5,
        linewidth=2.1,
    )
    ax.axhline(
        float(records[0]["condition_rmse"]["development"][condition]),
        color="#4c4c4c",
        linestyle="--",
        linewidth=1.1,
        alpha=0.75,
    )
    ax.scatter(
        [final_record["iteration"]],
        [final_record["condition_rmse"]["development"][condition]],
        marker="*",
        s=130,
        color="#d62728",
        edgecolors="black",
        linewidths=0.4,
        zorder=4,
    )
    ax.set_ylim(low - margin, high + margin)
    ax.set_title(CONDITION_LABELS[condition])
    ax.set_ylabel("RMSE (degrees)")
    ax.grid(True, alpha=0.18)
for ax in axes[-1]:
    ax.set_xlabel("Experiment #")
fig.suptitle("Development RMSE by PCPNet condition", fontsize=15)
plt.show()

## Accuracy-runtime trade-off

All measured candidates appear as a faint cloud. The purple line marks the measured Pareto frontier; retained milestones are overlaid in green and the frozen finalist is starred.


In [ ]:
measured_records = [
    record
    for record in records
    if record.get("dev_rmse") is not None and record.get("runtime_s") is not None
]
runtimes = np.array([float(record["runtime_s"]) for record in measured_records])
scores = np.array([float(record["dev_rmse"]) for record in measured_records])
pareto = np.array(
    [
        not np.any(
            (runtimes <= runtime)
            & (scores <= score)
            & ((runtimes < runtime) | (scores < score))
        )
        for runtime, score in zip(runtimes, scores, strict=True)
    ]
)
pareto_order = np.argsort(runtimes[pareto])

fig, ax = plt.subplots(figsize=(11, 7), layout="constrained")
ax.scatter(runtimes, scores, s=18, color="#9aa0a6", alpha=0.30, linewidths=0)
ax.plot(
    runtimes[pareto][pareto_order],
    scores[pareto][pareto_order],
    color="#7b3294",
    linewidth=2,
    marker="o",
    markersize=4,
    label="Measured Pareto frontier",
)
ax.scatter(
    [float(record["runtime_s"]) for record in keep_records],
    [float(record["dev_rmse"]) for record in keep_records],
    s=55,
    color=STATUS_COLORS["keep"],
    edgecolors="white",
    linewidths=0.7,
    label="Retained milestone",
    zorder=3,
)
ax.scatter(
    [float(final_record["runtime_s"])],
    [float(final_record["dev_rmse"])],
    marker="*",
    s=230,
    color="#d62728",
    edgecolors="black",
    linewidths=0.5,
    label="Frozen finalist",
    zorder=4,
)
ax.set(
    xlabel="Development runtime (seconds)",
    ylabel="Development RMSE (degrees, lower is better)",
    title="Accuracy-runtime trade-off",
)
ax.legend()
ax.grid(True, alpha=0.20)
plt.show()

## Final official-test comparison

The frozen finalist was evaluated once on all 108 entries and 540,000 official queries in `testset_all.txt`. The table below uses sign-invariant angular RMSE and PCPNet's mean of per-shape RMSE values, averaged equally across the six conditions.

Published rows come from the PFF-Net comparison table. The project result is directly comparable at the benchmark-protocol level, with one caveat: PCPNet's official test list includes the validation geometries. Negative deltas mean the project estimator has lower error.


In [ ]:
ours = (
    *[
        float(final_results["conditions"][condition]["rmse"])
        for condition in CONDITIONS
    ],
    float(final_results["rmse"]),
)
comparison = {
    "Final explainable estimator": ours,
    **PUBLISHED,
}
comparison_rows = [
    [method, *[f"{value:.2f}" for value in values]]
    for method, values in comparison.items()
]
display(
    Markdown(
        markdown_table(
            [
                "Method",
                "None",
                "Low",
                "Medium",
                "High",
                "Stripe",
                "Gradient",
                "Average",
            ],
            comparison_rows,
        )
    )
)

delta_pca = tuple(
    ours[index] - PUBLISHED["PCA (published)"][index] for index in range(7)
)
delta_pff = tuple(ours[index] - PUBLISHED["PFF-Net"][index] for index in range(7))
delta_rows = [
    ["Final - published PCA", *[f"{value:+.2f}" for value in delta_pca]],
    ["Final - PFF-Net", *[f"{value:+.2f}" for value in delta_pff]],
]
display(
    Markdown(
        "### Error deltas\n\n"
        + markdown_table(
            [
                "Comparison",
                "None",
                "Low",
                "Medium",
                "High",
                "Stripe",
                "Gradient",
                "Average",
            ],
            delta_rows,
        )
    )
)

In [ ]:
labels = [CONDITION_LABELS[condition] for condition in CONDITIONS] + ["Average"]
x = np.arange(len(labels))
methods = ("Final explainable estimator", "PCA (published)", "MSECNet", "PFF-Net")
colors = ("#1b9e77", "#7f8c8d", "#7570b3", "#d95f02")
width = 0.19

fig, ax = plt.subplots(figsize=(15, 7), layout="constrained")
for offset, method, color in zip((-1.5, -0.5, 0.5, 1.5), methods, colors, strict=True):
    ax.bar(
        x + offset * width,
        comparison[method],
        width,
        label=method,
        color=color,
        alpha=0.90,
    )
ax.set_xticks(x, labels)
ax.set_ylabel("Official-test RMSE (degrees, lower is better)")
ax.set_title("Final estimator versus published PCPNet references")
ax.legend(ncols=2)
ax.grid(axis="y", alpha=0.20)
plt.show()

## Interpretation

- The final explainable estimator achieves **14.36° average test RMSE**, improving over the published PCA reference (**16.25°**) by **1.89°**, or **11.6%** relative.
- It improves on published PCA in every condition. The largest absolute gains are on clean data, gradient density, low noise, and stripe density.
- Medium and high noise improve only slightly over published PCA. These remain the dominant limitation of a compact local PCA-derived method.
- The final result remains **4.98° behind PFF-Net** on average. The largest gaps are stripe density, high noise, gradient density, and clean data.
- The comparison supports the intended conclusion: robust local PCA substantially closes the classical-PCA gap while remaining explainable, but it does not reach learned-method SOTA.


In [ ]:
# Export the two most useful static artifacts.
progress_path = REPO_ROOT / "progress.png"
comparison_path = REPO_ROOT / "final-comparison.png"

fig, ax = plt.subplots(figsize=(16, 8), layout="constrained")
ax.scatter(
    iterations[measured_mask],
    np.clip(development[measured_mask], lower, upper),
    s=15,
    color=STATUS_COLORS["discard"],
    alpha=0.28,
    linewidths=0,
    label="Measured attempt",
)
ax.plot(
    iterations,
    running_best,
    color="#4c78a8",
    linewidth=1.8,
    label="Best measured so far",
)
ax.step(
    keep_iterations,
    keep_dev,
    where="post",
    color=STATUS_COLORS["keep"],
    linewidth=2.8,
    label="Retained frontier",
)
ax.scatter(
    keep_iterations,
    keep_dev,
    s=58,
    color=STATUS_COLORS["keep"],
    edgecolors="white",
    linewidths=0.8,
    zorder=3,
)
for record in keep_records:
    ax.annotate(
        f"#{record['iteration']}",
        (int(record["iteration"]), float(record["dev_rmse"])),
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
        fontsize=8,
    )
ax.scatter(
    [final_record["iteration"]],
    [final_record["dev_rmse"]],
    marker="*",
    s=230,
    color="#d62728",
    edgecolors="black",
    linewidths=0.5,
    label="Frozen finalist",
    zorder=4,
)
ax.set(
    xlabel="Experiment #",
    ylabel="Development RMSE (degrees, lower is better)",
    title=f"Normal-estimation search: {len(records)} completed experiments",
    ylim=(lower, upper),
)
ax.legend()
ax.grid(True, alpha=0.20)
fig.savefig(progress_path, dpi=180, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(15, 7), layout="constrained")
for offset, method, color in zip((-1.5, -0.5, 0.5, 1.5), methods, colors, strict=True):
    ax.bar(x + offset * width, comparison[method], width, label=method, color=color)
ax.set_xticks(x, labels)
ax.set_ylabel("Official-test RMSE (degrees, lower is better)")
ax.set_title("Final estimator versus published PCPNet references")
ax.legend(ncols=2)
ax.grid(axis="y", alpha=0.20)
fig.savefig(comparison_path, dpi=180, bbox_inches="tight")
plt.close(fig)

progress_path, comparison_path